### Instituto Tecnológico de la Laguna  
### Inteligencia Artificial  
### Ing. Sistemas Computacionales  

Presenta:   
- Mayra Paloma Fernandez De la Vara  
- Alondra Victoria Olvera Martínez  
- Iván Alonso Gallegos Herrera  
- Victor Manuel Silva de León  
- Hugo Ivan Montaña Rivera  
- Alan Gerardo Casas Chávez



## Sistema inteligente de detección de micro-paradas en una línea de producción.

**Descripción**  
En muchas fábricas (automotriz, alimentos, electrónica), existen micro-paradas: interrupciones muy cortas (1–10 segundos) que no se registran como fallas, pero reducen significativamente la productividad. Estas micro-paradas: No siempre son detectadas por sistemas tradicionales, se mezclan con ruido en los sensores y generan pérdidas económicas importantes.  

El proyecto consiste en _diseñar e implementar un sistema basado en redes neuronales que detecte las micro-paradas en tiempo real, clasifique la posible causa y prediga cuándo ocurrirá la siguiente_. Para esto se debe crear el dataset (NOTA: se incluye el código para generarlo y no debe utilizarse otro ya  que contiene: señales de sensores industriales (vibración, corriente eléctrica, velocidad de motor y temperatura). las señales deben tener etiquetas como normal, micro-parada y tipo de causa (falta de material, desalineación, error humano o falla mecánica leve). En la generación de los datos debe asegurarse que contenga ruido, estén incompletos y desbalaceandos para simular datos en la vida real.

**Restricciones**: no se puede usar frameworks de alto nivel como keras. Tampoco el uso de forward pass, backpropagation o funciones de pérdida. SÍ se puede utilizar numpy

**Arquitectura obligatoria**: se debe combinar una red neuronal para series temporales (RNN, LSTM o GRU (implementadas manualmente)), capa de atención simple opcional pero recomendada. El sistema debe contener los siguientes módulos: _**a) detección**_ (clasificación binaria) entrada: ventana de tiempo. Salida: ¿micro-parada? _**b) clasificación de causa**_ (multiclase) solo si detecta micro-parada y _**c) predicción temporal**_ (predecir en cuántos segundos ocurrirá la siguiente micro-parada

**A entregar**: Código completo sin frameworks (notebook), documentación (diseño del modelo, problemas encontrados, justificación de decisiones), visualizaciones (señales vs. predicciones), Simulación del impacto (cuánto downtime se reduce? Downtime es el tiempo en que una máquina se encuentra inactiva)

En general al realizar el proyecto están integrando series temporales, clasificaciones y predicciones. Limpieza de datos, uso de clases desbalanceadas y señales.

In [ ]:
import numpy as np

## Configuración y Estado Inicial
Aseguramos que los números "aleatorios" sean siempre los mismos cada vez que se corra el código.

In [ ]:
np.random.seed(42)

In [ ]:
# Parámetros generales
TOTAL_TIME = 20000 # número de pasos de tiempo
MICRO_STOP_PROB = 0.003 # eventos raros (~0.3%)
MAX_LAG = 10 # desfase temporal máximo
NUM_SENSORS = 4

Creación de 4 variables sensorizadas (vibración, corriente, velocidad y temperatura) usando distribuciones normales.
                                      

In [ ]:
# Variables
vibration = np.random.normal(0, 0.2, TOTAL_TIME)
current = np.random.normal(5, 0.5, TOTAL_TIME)
speed = np.random.normal(100, 2, TOTAL_TIME)
temperature = np.random.normal(70, 1, TOTAL_TIME)

Usaremos `plt.hist` para poder visualizar la distribución de los datos, con esto podemos identificar la "forma" de la operación, detección de ruido y dispersión y localizar valores atípicos (outliers).

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(2, 2, figsize=(10, 8))

axs[0, 0].hist(vibration, bins=50, color='blue', edgecolor='black')
axs[0, 0].set_title('Vibración (Media 0)')

axs[0, 1].hist(current, bins=50, color='green', edgecolor='black')
axs[0, 1].set_title('Corriente (Media 5)')

axs[1, 0].hist(speed, bins=50, color='red', edgecolor='black')
axs[1, 0].set_title('Velocidad (Media 100)')

axs[1, 1].hist(temperature, bins=50, color='orange', edgecolor='black')
axs[1, 1].set_title('Temperatura (Media 70)')

plt.tight_layout()
plt.show()

In [ ]:
# Labels
micro_stop = np.zeros(TOTAL_TIME)
cause = np.zeros(TOTAL_TIME) # 0 = normal

Definición de cuatro tipos de problemas que pueden ocurrir. Cada uno afectará a los sensores de forma distinta:
1. **Falta de material**: Baja la velocidad y la corriente.
2. **Desalineación**: Aumenta mucho la vibración.
3. **Error humano**: Provoca fluctuaciones erráticas en la velocidad.
4. **Fallo mecánico**: Sube la temperatura y la vibración gradualmente.

In [ ]:
CAUSES = {
    1: "falta_material",
    2: "desalineacion",
    3: "error_humano",
    4: "fallo_mecanico"
}

In [ ]:
def add_noise(signal, noise_level=0.1):
    return signal + np.random.normal(0, noise_level, len(signal))

## Motor de inyección
El lag elige un desfase de entre 2 y 10 pasos. Esto simula que los sensores empiezan a fallar **antes** de que la máquina se detenga por completo.

Dependiendo de la causa, altera los valores de los sensores usando `np.linspace` para crear un efecto de degradación progresiva.

En el momento exacto del fallo (`t` a `t+3`), la velocidad cae a 0 y la corriente se reduce a la mitad. 

In [ ]:
def inject_micro_stop(t):
    lag = np.random.randint(2, MAX_LAG)

    cause_type = np.random.choice(list(CAUSES.keys()))

    # Marca evento
    micro_stop[t] = 1
    cause[t] = cause_type

    # Simular causa antes (desfase)
    start = max(0, t - lag)

    if cause_type == 1:  # falta material
        speed[start:t] -= np.linspace(0, 20, t - start)
        current[start:t] -= np.linspace(0, 1, t - start)

    elif cause_type == 2:  # desalineación
        vibration[start:t] += np.linspace(0, 3, t - start)

    elif cause_type == 3:  # error humano
        speed[start:t] += np.random.normal(0, 10, t - start)

    elif cause_type == 4:  # fallo mecánico leve
        temperature[start:t] += np.linspace(0, 10, t - start)
        vibration[start:t] += np.linspace(0, 2, t - start)

    # Efecto durante micro-parada
    speed[t:t+3] = 0
    current[t:t+3] *= 0.5

## Generación de Eventos y Post-procesamiento
El bucle principal recorre los 20,000 pasos de tiempo. Si un número aletorio es menor a 0.003 (0.3% de probabilidad), activa una falla.

In [ ]:
# Generar eventos
for t in range(20, TOTAL_TIME - 20):
    if np.random.rand() < MICRO_STOP_PROB:
        inject_micro_stop(t)

Usa la función `add_noise` para que los datos no sean perfectos. Esto simula las interferencias eléctricas o vibraciones naturales de la fábrica.

In [ ]:
# Añadir ruido global (simular sensores reales)
vibration = add_noise(vibration, 0.3)
current = add_noise(current, 0.2)
speed = add_noise(speed, 1.0)
temperature = add_noise(temperature, 0.5)

In [ ]:
# Introducir datos faltantes (missing values)
def add_missing(signal, missing_prob=0.01):
    mask = np.random.rand(len(signal)) < missing_prob
    signal[mask] = np.nan
    return signal

Finalmente, el código junta todas las columnas (sensores + etiquetas de falla) en una sola matriz y la guarda como un archivo CSV llamado `industrial_dataset.csv`

In [ ]:
vibration = add_missing(vibration)
current = add_missing(current)
speed = add_missing(speed)
temperature = add_missing(temperature)

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10, 8))

axs[0, 0].hist(vibration, bins=50, color='blue', edgecolor='black')
axs[0, 0].set_title('Vibración (Media 0)')

axs[0, 1].hist(current, bins=50, color='green', edgecolor='black')
axs[0, 1].set_title('Corriente (Media 5)')

axs[1, 0].hist(speed, bins=50, color='red', edgecolor='black')
axs[1, 0].set_title('Velocidad (Media 100)')

axs[1, 1].hist(temperature, bins=50, color='orange', edgecolor='black')
axs[1, 1].set_title('Temperatura (Media 70)')

plt.tight_layout()
plt.show()

In [ ]:
# Dataset final
dataset = np.column_stack([
    vibration,
    current,
    speed,
    temperature,
    micro_stop,
    cause
])

In [ ]:
print("Dataset shape:", dataset.shape)

In [ ]:
# Guardar
np.savetxt("industrial_dataset.csv", dataset, delimiter=",",
           header="vibration,current,speed,temperature,micro_stop,cause",
           comments='')

| Columna | Descripción |
| :--- | :--- | 
| Vibration | Sensor de vibración (afectado por desalineación y fallos mecánicos). |
| Current | Consumo eléctrico (cae en paradas y falta de material) |
| Speed | Velocidad de operación (va a 0 en paradas) |
| Temperature | Calor de la máquina (sube en fallos mecánicos) |
| Micro_stop | Etiqueta (Target): 0 si todo está bien, 1 si hay parada |
| Cause | Categoría: Del 1 al 4 según el tipo de problema |

## Limpieza de datos
Lo primero que se debe realizar con nuestros datos de prueba es una limpieza, esto nos ayuda a garantizar la calidad, coherencia y fiabilidad de la información. En este caso *limpiaremos los outliers de nuestra información.*

In [ ]:
def clean_data(data, threshold=3):
    """
    Solo limpia outliers y marca NaNs en las columnas de sensores.
    """
    cleaned_data = data.copy()
    
    # Solo iteramos por las columnas de sensores (0, 1, 2, 3)
    for i in range(NUM_SENSORS):
        col = cleaned_data[:, i]
        mean = np.nanmean(col)
        std = np.nanstd(col)
        
        lower_bound = mean - threshold * std
        upper_bound = mean + threshold * std
        
        outliers_mask = (col < lower_bound) | (col > upper_bound)
        cleaned_data[outliers_mask, i] = np.nan
        
    print(f"Limpieza: Se procesaron las {NUM_SENSORS} columnas de sensores.")
    return cleaned_data

## Imputación de datos
La imputación nos sirve para *reemplazar valores faltantes, nulos o atípicos en un conjunto de datos con valores simulados o estimados*, esto nos garantiza que el dataset esté completo para su análisis sin perder información valiosa.

In [ ]:
def impute_data(data):
    """
    Aplica Forward Fill solo a las columnas de sensores.
    """
    filled_data = data.copy()
    
    for i in range(NUM_SENSORS):
        col = filled_data[:, i]
        mask = np.isnan(col)
        
        # Si no hay NaNs en esta columna, saltamos a la siguiente
        if not np.any(mask):
            continue
            
        idx = np.arange(len(col))
        idx[mask] = 0
        filled_idx = np.maximum.accumulate(idx)
        
        if np.isnan(col[0]):
            # Buscamos el primer valor real para el caso del primer elemento NaN
            valid_indices = np.where(~mask)[0]
            if len(valid_indices) > 0:
                col[0] = col[valid_indices[0]]
                idx[mask] = 0
                filled_idx = np.maximum.accumulate(idx)

        filled_data[:, i] = col[filled_idx]
        
    print(f"Imputación: NaNs rellenados en las {NUM_SENSORS} columnas de sensores.")
    return filled_data

## Normalización de datos
Nos ayudará a *transformar nuestros valores númericos de diferentes features a una escala común*, sin distorsionar las diferencias en los rangos de valores ni perder información.

In [ ]:
def normalize_date(data):
    """
    Normaliza solo los sensores [0, 1]. Las etiquetas se quedan igual.
    """
    normalized_data = data.copy()
    sensors = normalized_data[:, :NUM_SENSORS]
    
    min_val = np.min(sensors, axis=0)
    max_val = np.max(sensors, axis=0)
    
    # Evitar división por cero si un sensor es constante
    range_val = max_val - min_val
    range_val[range_val == 0] = 1 
    
    normalized_data[:, :NUM_SENSORS] = (sensors - min_val) / range_val
    
    print(f"Normalización: Sensores escalados. Etiquetas (Micro Stop/Cause) intactas.")
    return normalized_data

In [ ]:
dataset_cleaned = clean_data(dataset)
dataset_imputed = impute_data(dataset_cleaned)
dataset_normalized = normalize_date(dataset_imputed)

In [ ]:
np.savetxt("dataset_preproc.csv", dataset_normalized, delimiter=",",
           header="vibration,current,speed,temperature,micro_stop,cause",
           comments='')